# Installation and tiny synthetic smoke test
This example checks the actual solver on synthetic data. It is not a biological benchmark or paper result.


## How to run and what to expect

Install the environment using [README.md](README.md), select **Python (raftup_revision)**, then run all cells in order. **No downloads, input folder or pair manifest are required.** The code creates a four-observation synthetic pair in memory and checks that its alignment support is the identity. A successful run prints coverage A = 1.0 and coverage B = 1.0 without an assertion error. This notebook does not save an alignment; the command-line runner can save its executed notebook and status report.


In [ ]:
from pathlib import Path
import os, json
import numpy as np
import raftup
from raftup import run_automated_alignment, run_manual_alignment
from raftup.experiments import load_pair, save_alignment, coverage
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file()), Path.cwd())
DATA = Path(os.environ.get('RAFTUP_TUTORIAL_DATA', str(REPO / 'data/revision/smoke'))).expanduser().resolve()
OUT = Path(os.environ.get('RAFTUP_OUTPUT_ROOT', str(REPO / 'outputs/revision/00_installation_smoke_test'))).expanduser().resolve()
print('Inputs:', DATA, '\nOutputs:', OUT)
print('RAFT-UP:', raftup.__version__, raftup.__file__)


In [ ]:
import anndata as ad
from scipy.spatial.distance import cdist
xy = np.array([[0.,0.],[10.,0.],[0.,10.],[10.,10.]])
a = ad.AnnData(np.eye(4)); b = ad.AnnData(np.eye(4))
a.obsm['spatial'] = xy.copy(); b.obsm['spatial'] = xy.copy()
cost = cdist(np.eye(4), np.eye(4))
P, decision = run_manual_alignment(a, b, cost, alignment_mode='window',
    downsampling_ratio=1., feature_cutoff_downsample=.1,
    spatial_cutoff_downsample_um=1., technology='dlpfc', pixels_per_100um=100,
    random_state=42)
assert np.array_equal(P > 0, np.eye(4, dtype=bool))
print(coverage(P))
